In [1]:
import pandas as pd
import numpy as np

columns = [
    "engine_id", "cycle",
    "op_setting_1", "op_setting_2", "op_setting_3"
] + [f"sensor_{i}" for i in range(1, 22)]

train_df = pd.read_csv(
    "../dataset/train_FD001.txt",
    sep=r"\s+",
    header=None
)

train_df.columns = columns

print(train_df.shape)
print(train_df.head())

(20631, 26)
   engine_id  cycle  op_setting_1  op_setting_2  op_setting_3  sensor_1  \
0          1      1       -0.0007       -0.0004         100.0    518.67   
1          1      2        0.0019       -0.0003         100.0    518.67   
2          1      3       -0.0043        0.0003         100.0    518.67   
3          1      4        0.0007        0.0000         100.0    518.67   
4          1      5       -0.0019       -0.0002         100.0    518.67   

   sensor_2  sensor_3  sensor_4  sensor_5  ...  sensor_12  sensor_13  \
0    641.82   1589.70   1400.60     14.62  ...     521.66    2388.02   
1    642.15   1591.82   1403.14     14.62  ...     522.28    2388.07   
2    642.35   1587.99   1404.20     14.62  ...     522.42    2388.03   
3    642.35   1582.79   1401.87     14.62  ...     522.86    2388.08   
4    642.37   1582.85   1406.22     14.62  ...     522.19    2388.04   

   sensor_14  sensor_15  sensor_16  sensor_17  sensor_18  sensor_19  \
0    8138.62     8.4195       0.0

In [2]:
train_df["max_cycle"] = (
    train_df.groupby("engine_id")["cycle"].transform("max")
)

train_df["RUL"] = (
    train_df["max_cycle"] - train_df["cycle"]
)

print(train_df[["engine_id", "cycle", "RUL"]].head())

   engine_id  cycle  RUL
0          1      1  191
1          1      2  190
2          1      3  189
3          1      4  188
4          1      5  187


In [3]:
feature_columns = [
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_19",
    "sensor_20",
    "sensor_21"
]

print(len(feature_columns))

22


In [4]:
from sklearn.model_selection import train_test_split

engine_ids = train_df["engine_id"].unique()

train_engines, val_engines = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42
)

train_data = train_df[
    train_df["engine_id"].isin(train_engines)
]

val_data = train_df[
    train_df["engine_id"].isin(val_engines)
]

print(train_data["engine_id"].nunique())
print(val_data["engine_id"].nunique())

80
20


In [19]:
MAX_LEN = 100

def create_training_sequences(data, features):
    X, y = [], []

    for _, engine in data.groupby("engine_id"):
        engine = engine.sort_values("cycle")

        values = engine[features].values
        rul = engine["RUL"].values

        for i in range(len(engine)):
            start = max(0, i - MAX_LEN + 1)

            sequence = values[start:i+1]

            if len(sequence) < MAX_LEN:
                padding = np.zeros(
                    (MAX_LEN - len(sequence), len(features))
                )
                sequence = np.vstack([padding, sequence])

            X.append(sequence)
            y.append(rul[i])

    return np.array(X), np.array(y)

In [20]:
X_train, y_train = create_training_sequences(
    train_data, feature_columns
)

X_val, y_val = create_training_sequences(
    val_data, feature_columns
)

print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

print("RUL range:", y_train.min(), y_train.max())

(16561, 100, 22)
(16561,)
(4070, 100, 22)
(4070,)
RUL range: 0 361


In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on real training data
scaler.fit(train_data[feature_columns])

# Scale train and validation sequences
X_train_scaled = scaler.transform(
    X_train.reshape(-1, 22)
).reshape(X_train.shape)

X_val_scaled = scaler.transform(
    X_val.reshape(-1, 22)
).reshape(X_val.shape)

c:\Users\Lenovo\Desktop\aeroguardian\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Lenovo\Desktop\aeroguardian\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [22]:
X_train_scaled[X_train == 0] = -999
X_val_scaled[X_val == 0] = -999

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense, Dropout

lstm_final = Sequential([
    Masking(mask_value=-999, input_shape=(100, 22)),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

lstm_final.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_final.summary()

c:\Users\Lenovo\Desktop\aeroguardian\venv\Lib\site-packages\keras\src\layers\core\masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_2 (Masking)             │ (None, 100, 22)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        22,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,385 (95.25 KB)

 Trainable params: 24,385 (95.25 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = lstm_final.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 72s 64ms/step - loss: 3791.3152 - mae: 41.6418 - val_loss: 1043.5552 - val_mae: 23.5807
Epoch 2/30
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 63s 61ms/step - loss: 1629.8271 - mae: 28.2382 - val_loss: 916.9145 - val_mae: 22.3768
Epoch 3/30
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 61s 59ms/step - loss: 1483.5126 - mae: 26.5767 - val_loss: 860.8053 - val_mae: 22.0153
Epoch 4/30
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 65s 63ms/step - loss: 1310.4716 - mae: 24.5472 - val_loss: 953.9365 - val_mae: 23.3393
Epoch 5/30
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 65s 63ms/step - loss: 1042.8477 - mae: 22.0024 - val_loss: 1081.4088 - val_mae: 23.9942
Epoch 6/30
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 66s 63ms/step - loss: 709.2999 - mae: 18.3146 - val_loss: 1266.6917 - val_mae: 26.0385
Epoch 7/30
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 66s 63ms/step - loss: 549.3634 - mae: 16.1004 - val_loss: 1212.6262 - val_mae: 25.1812
Epoch 8/30
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 67s 65ms/step - loss: 459.4763 - mae: 14.8016 - val_l

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

val_pred = lstm_final.predict(X_val_scaled).ravel()

print("MAE:", mean_absolute_error(y_val, val_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, val_pred)))
print("R²:", r2_score(y_val, val_pred))

128/128 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step
MAE: 22.01532554626465
RMSE: 29.339479438308675
R²: 0.800285279750824


In [26]:
lstm_final.save("../lstm_final.keras")

import joblib
joblib.dump(scaler, "../scaler_final.pkl")

['../scaler_final.pkl']

In [29]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# --------------------------------------------------
# 1. Load training data
# --------------------------------------------------

columns = [
    "engine_id",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3"
] + [f"sensor_{i}" for i in range(1, 22)]

train_df = pd.read_csv(
    "../dataset/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

# Remove constant sensors
train_df = train_df.drop(
    columns=["sensor_5", "sensor_16", "sensor_18"]
)

# Same 22 features used by our original LSTM
feature_cols = [
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_19",
    "sensor_20",
    "sensor_21"
]

# Original uncapped RUL
max_cycle = train_df.groupby(
    "engine_id"
)["cycle"].transform("max")

train_df["RUL"] = max_cycle - train_df["cycle"]


# --------------------------------------------------
# 2. Same 80/20 ENGINE split
# --------------------------------------------------

engine_ids = sorted(train_df["engine_id"].unique())

train_engines, val_engines = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42
)

print("Validation engines:", val_engines)


# --------------------------------------------------
# 3. Load ORIGINAL 100-cycle LSTM + scaler
# --------------------------------------------------

model = tf.keras.models.load_model(
    "../lstm100.keras"
)

scaler = joblib.load(
    "../scaler100.pkl"
)

SEQ_LEN = 100


# --------------------------------------------------
# 4. Create ONE test-like sample per validation engine
# --------------------------------------------------

X_val_testlike = []
y_val_true = []
val_engine_ids = []
cutoffs = []

for engine_id in val_engines:

    engine = train_df[
        train_df["engine_id"] == engine_id
    ].sort_values("cycle")

    max_cyc = engine["cycle"].max()

    # Pretend NASA stopped this engine at 80% of its life
    cutoff = int(0.8 * max_cyc)

    available = engine[
        engine["cycle"] <= cutoff
    ]

    values = available[feature_cols].values

    # Scale using TRAINING scaler
    values_scaled = scaler.transform(values)

    # Take latest 100 cycles
    if len(values_scaled) >= SEQ_LEN:

        sequence = values_scaled[-SEQ_LEN:]

    else:

        pad_len = SEQ_LEN - len(values_scaled)

        # Repeat first available scaled cycle
        padding = np.repeat(
            values_scaled[0:1],
            pad_len,
            axis=0
        )

        sequence = np.vstack([
            padding,
            values_scaled
        ])

    X_val_testlike.append(sequence)

    # True RUL at the artificial cutoff
    y_val_true.append(
        max_cyc - cutoff
    )

    val_engine_ids.append(engine_id)
    cutoffs.append(cutoff)


X_val_testlike = np.array(X_val_testlike)
y_val_true = np.array(y_val_true)


print(
    "Validation input shape:",
    X_val_testlike.shape
)

print(
    "Validation target shape:",
    y_val_true.shape
)


# --------------------------------------------------
# 5. Predict ONCE per validation engine
# --------------------------------------------------

y_val_pred = model.predict(
    X_val_testlike,
    verbose=0
).flatten()


# --------------------------------------------------
# 6. Metrics
# --------------------------------------------------

mae = mean_absolute_error(
    y_val_true,
    y_val_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_val_true,
        y_val_pred
    )
)

r2 = r2_score(
    y_val_true,
    y_val_pred
)

print("\nTest-like validation results:")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


# --------------------------------------------------
# 7. See individual predictions
# --------------------------------------------------

results = pd.DataFrame({
    "engine_id": val_engine_ids,
    "cutoff_cycle": cutoffs,
    "true_RUL": y_val_true,
    "pred_RUL": y_val_pred
})

print("\nPredictions:")
print(results)

Validation engines: [np.int64(84), np.int64(54), np.int64(71), np.int64(46), np.int64(45), np.int64(40), np.int64(23), np.int64(81), np.int64(11), np.int64(1), np.int64(19), np.int64(31), np.int64(74), np.int64(34), np.int64(91), np.int64(5), np.int64(77), np.int64(78), np.int64(13), np.int64(32)]
Validation input shape: (20, 100, 22)
Validation target shape: (20,)

Test-like validation results:
MAE : 3.512512683868408
RMSE: 4.027931444298761
R²  : 0.7668597102165222

Predictions:
    engine_id  cutoff_cycle  true_RUL   pred_RUL
0          84           213        54  50.782345
1          54           205        52  53.625446
2          71           166        42  36.463963
3          46           204        52  49.572018
4          45           126        32  35.266685
5          40           150        38  33.791882
6          23           134        34  31.943552
7          81           192        48  47.773983
8          11           192        48  54.397427
9           1           